# Анализ рынка видеоигр

## Цели и задачи проекта

Цель проекта: изучение данных, содержащих информацию о продажах игр разных жанров и платформ.

Задачи:
- загрузка данных;
- проведение предобработки данных, поиск пропусков и дубликатов;
- фильтрация данных по времени выхода игры;
- категоризация данных по оценкам пользователей и критиков;
- определение лидирующих платформ по количеству игр, выпущенных за весь актуальный период.

### Описание данных

Данные представлены в форме CSV файла и содержат информацию о продажах игр разных жанров и платформ, а также пользовательские и экспертные оценки игр:

Name — название игры.

Platform — название платформы.

Year of Release — год выпуска игры.

Genre — жанр игры.

NA sales — продажи в Северной Америке (в миллионах проданных копий).

EU sales — продажи в Европе (в миллионах проданных копий).

JP sales — продажи в Японии (в миллионах проданных копий).

Other sales — продажи в других странах (в миллионах проданных копий).

Critic Score — оценка критиков (от 0 до 100).

User Score — оценка пользователей (от 0 до 10).

Rating — рейтинг организации ESRB (англ. Entertainment Software Rating Board).

### Содержимое проекта

1) Цели и задачи проекта

2) Загрузка и знакомство с данными датасета

3) Проверка ошибок в данных и их предобработка 

4) Фильтрация данных по времени выхода игры

5) Категоризация информации по оценкам пользователей и критиков 

6) Подведение итогов

## Загрузка данных и знакомство с ними

In [1]:
# Загружаем библиотеку pandas:
import pandas as pd

In [2]:
# Загружаем данные датасета: 
df = pd.read_csv ('https:/.../new_games.csv')

In [3]:
# Создаем копию
df_origin = df.copy()

In [4]:
# Выводим первые 5 строк датафрейма:
df.head()

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


In [5]:
# Получаем информацию о датафрейме
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


Датафрейм состоит из 11 столбцов и содержит 16956 строк. Названия и значения столбцов соответствуют описанию. 
Данные содержат пропуски в следующих столбцах: Name, Year of Release, Genre, Critic Score, User Score, Rating.
Ряд столбцов не содержит ожидаемый тип данных: Year of Release (float64), EU sales (object), JP sales (object), User Score (object).

---

## Проверка ошибок в данных и их предобработка

In [6]:
# Выводим названия всех столбцов:
df.columns

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')

In [7]:
# Приводим названия столбцов к стилю snake case:
df.columns = df.columns.str.lower().str.replace(' ', '_')
df.columns

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='object')

### Типы данных

Некорректный тип данных встречается в столбцах Year of Release (float64), EU sales (object), JP sales (object), User Score (object). При выгрузке данных в датафрейм pandas самостоятельно определяет подходящий тип данных. Столбцы EU sales, JP sales, User Score относятся к типу данных object, однако, эти поля должны содержать числа. Следовательно, в этих столбцах есть данные, которые не относятся к числовым типам.


In [8]:
# Находим значения в столбце EU sales, которые не относятся к числовым
non_numeric_eu_sales = df['eu_sales'][pd.to_numeric(df['eu_sales'], errors='coerce').isna()]
non_numeric_eu_sales

446     unknown
802     unknown
1131    unknown
1132    unknown
1394    unknown
1612    unknown
Name: eu_sales, dtype: object

In [9]:
# Находим значения в столбце JP sales, которые не относятся к числовым
non_numeric_jp_sales = df['jp_sales'][pd.to_numeric(df['jp_sales'], errors='coerce').isna()]
non_numeric_jp_sales

467     unknown
819     unknown
1379    unknown
4732    unknown
Name: jp_sales, dtype: object

In [10]:
# Находим значения в столбце User Score, которые не относятся к числовым
non_numeric_user_score = df['user_score'][pd.to_numeric(df['user_score'], errors='coerce').isna() & df['user_score'].notna()]
non_numeric_user_score

119      tbd
302      tbd
522      tbd
647      tbd
659      tbd
        ... 
16935    tbd
16937    tbd
16938    tbd
16945    tbd
16947    tbd
Name: user_score, Length: 2464, dtype: object

Выявленные строковые значения означают отсутствие данных в столбцах и не несут полезной для анализа информации. Следовательно, данные значения можно поменять на пропуски.

In [11]:
# Приводим значения в столбце EU sales, к числовому типу данных, меняя строковые значения на пропуски:
df['eu_sales'] = pd.to_numeric(df['eu_sales'], errors='coerce')
df['eu_sales'].dtype

dtype('float64')

In [12]:
# Приводим значения в столбце JP sales, к числовому типу данных, меняя строковые значения на пропуски:
df['jp_sales'] = pd.to_numeric(df['jp_sales'], errors='coerce')
df['jp_sales'].dtype

dtype('float64')

In [13]:
# Приводим значения в столбце User Score, к числовому типу данных, меняя строковые значения на пропуски:
df['user_score'] = pd.to_numeric(df['user_score'], errors='coerce')
df['user_score'].dtype

dtype('float64')

### Наличие пропусков в данных

In [14]:
# Считаем количество пропусков в датафрейме в абсолютных значениях: 
мissing_value_count =  df.isna().sum()
мissing_value_count

name                  2
platform              0
year_of_release     275
genre                 2
na_sales              0
eu_sales              6
jp_sales              4
other_sales           0
critic_score       8714
user_score         9268
rating             6871
dtype: int64

In [15]:
# Считаем количество пропусков в датафрейме в относительных значениях: 
percent_of_missing_values =  df.isna().mean()
percent_of_missing_values 

name               0.000118
platform           0.000000
year_of_release    0.016218
genre              0.000118
na_sales           0.000000
eu_sales           0.000354
jp_sales           0.000236
other_sales        0.000000
critic_score       0.513918
user_score         0.546591
rating             0.405225
dtype: float64

По результату изучения столбцов датафрейма пропуски обнаружены в следующих полях:

-  Name - 2 (0.0001)

- Year of Release - 275 (0.016) 

- Genre - 2 (0.0001)

- EU sales - 6 (0.0004)

- JP sales - 4 (0.0002) 

- Critic Score - 8714 (0.51)

- User Score - 9268 (0.55) 

- Rating - 6871 (0.41)  

Количество пропусков в столбцах Name и Genre незначительно, а сами данные не используются в анализе, поэтому эти пропуски можно оставить без изменений.

В столбце Year of Release содержится данные о годе выхода игры. В этой связи, попытка заменить пропуски каким-то обобщённым значением приведёт к заметному искажению итогового результата. Так как столбец Year of Release в дельнейшем будет использован для фильтрации данных, а у пустых значений невозможно однозначно определить год, пропуски в этом поле можно удалить. 

Столбцы EU sales и JP sales содержат незначительное количество пропусков, а их влияние на дальнейший анализ будет несущественным. Поэтому данные пропуски можно заменить на значение-индикатор 0.

Более половины значений в столбцах Critic Score и User Score содержат пропуски. Оба этих столбца будут использоваться для категоризации данных, поэтому пропуски необходимо заменить на среднее значение. Учитывая большое количество пропусков и возможный большой размах и разброс значений, необходимо заменить пропуски на медианное значение в зависимости от названия платформы и года выхода игры.

Несмотря на то, что поле Rating содержит довольно большое количество пропусков, значения данного столбца не используются в анализ, а следовательно, данные пропуски можно игнорировать.


In [16]:
# Удаляем строки с пропусками в столбце Year of Release:
df = df.dropna(subset=['year_of_release'])

In [17]:
# Поменяем тип данных в столбце Year of Release для удобства восприятия:
df['year_of_release'] = df['year_of_release'].astype('int64')
df['year_of_release'].dtype

dtype('int64')

In [18]:
# Меняем пропуски в столбце EU sales на значение индикатор:
df['eu_sales'] = df['eu_sales'].fillna(0)
df['eu_sales'].isna().sum()

0

In [19]:
# Меняем пропуски в столбце JP sales на значение индикатор:
df['jp_sales'] = df['jp_sales'].fillna(0)
df['jp_sales'].isna().sum()

0

In [20]:
# Меняем пропуски в столбце Critic Score на медианное значение в зависимости от названия платформы и года выхода игры:
df['critic_score'] = df['critic_score'].fillna(
    df.groupby(['platform', 'year_of_release'])['critic_score'].transform('median')
)

In [21]:
# Меняем пропуски в столбце User Score на медианное значение в зависимости от названия платформы и года выхода игры:
df['user_score'] = df['user_score'].fillna(
    df.groupby(['platform', 'year_of_release'])['user_score'].transform('median')
)

In [22]:
# Проверяем пропуски после изменений (абсолютные значения):
df.isna().sum()


name                  2
platform              0
year_of_release       0
genre                 2
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score       1480
user_score         1256
rating             6780
dtype: int64

In [23]:
# Проверяем пропуски после изменений (относительные значения):
df.isna().mean()


name               0.000120
platform           0.000000
year_of_release    0.000000
genre              0.000120
na_sales           0.000000
eu_sales           0.000000
jp_sales           0.000000
other_sales        0.000000
critic_score       0.088724
user_score         0.075295
rating             0.406450
dtype: float64

После проведённых изменеий количество пропусков существенно сократилось, однако, в столбцах Critic Score и User Score остаётся большое количество пропусков, т.к. не для всех сочетаний года выхода игры и платформы нашлось медианное значение. Замена такого количества оставшихся пропусков на общее среднее значение приведёт к искажению результатов категоризации, а их удаление - к заметному искажению количества игр. Наилучшем решением для оставшихся пропусков будет заменить их на индикатор -1 и при проведении категоризации для таких значений добавить дополнительную группу "Нет оценки".

In [24]:
# Меняем оставшиеся пропуски в столбце User Score на значение-индикатор -1:
df['user_score'] = df['user_score'].fillna(-1)

In [25]:
# Меняем оставшиеся пропуски в столбце Critic Score на значение-индикатор -1:
df['critic_score'] = df['critic_score'].fillna(-1)

In [26]:
# Проверяем пропуски после изменений
df.isna().sum()

name                  2
platform              0
year_of_release       0
genre                 2
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score          0
user_score            0
rating             6780
dtype: int64

In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16681 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16679 non-null  object 
 1   platform         16681 non-null  object 
 2   year_of_release  16681 non-null  int64  
 3   genre            16679 non-null  object 
 4   na_sales         16681 non-null  float64
 5   eu_sales         16681 non-null  float64
 6   jp_sales         16681 non-null  float64
 7   other_sales      16681 non-null  float64
 8   critic_score     16681 non-null  float64
 9   user_score       16681 non-null  float64
 10  rating           9901 non-null   object 
dtypes: float64(6), int64(1), object(4)
memory usage: 1.5+ MB


### Явные и неявные дубликаты в данных

In [28]:
# Выводим уникальные значения столбца Platform
df[('platform')].unique()

array(['Wii', 'NES', 'GB', 'DS', 'X360', 'PS3', 'PS2', 'SNES', 'GBA',
       'PS4', '3DS', 'N64', 'PS', 'XB', 'PC', '2600', 'PSP', 'XOne',
       'WiiU', 'GC', 'GEN', 'DC', 'PSV', 'SAT', 'SCD', 'WS', 'NG', 'TG16',
       '3DO', 'GG', 'PCFX'], dtype=object)

In [29]:
# Выводим уникальные значения столбца Genre
df[('genre')].unique()

array(['Sports', 'Platform', 'Racing', 'Role-Playing', 'Puzzle', 'Misc',
       'Shooter', 'Simulation', 'Action', 'Fighting', 'Adventure',
       'Strategy', nan, 'MISC', 'ROLE-PLAYING', 'RACING', 'ACTION',
       'SHOOTER', 'FIGHTING', 'SPORTS', 'PLATFORM', 'ADVENTURE',
       'SIMULATION', 'PUZZLE', 'STRATEGY'], dtype=object)

In [30]:
# Приводим значения к нижнему регистру 
df[('genre')] = df[('genre')].str.lower()
df[('genre')].unique()

array(['sports', 'platform', 'racing', 'role-playing', 'puzzle', 'misc',
       'shooter', 'simulation', 'action', 'fighting', 'adventure',
       'strategy', nan], dtype=object)

In [31]:
# Выводим уникальные значения столбца Year of release
df[('year_of_release')].unique()

array([2006, 1985, 2008, 2009, 1996, 1989, 1984, 2005, 1999, 2007, 2010,
       2013, 2004, 1990, 1988, 2002, 2001, 2011, 1998, 2015, 2012, 2014,
       1992, 1997, 1993, 1994, 1982, 2016, 2003, 1986, 2000, 1995, 1991,
       1981, 1987, 1980, 1983])

In [32]:
# Выводим уникальные значения столбца Rating
df[('rating')].unique()

array(['E', nan, 'M', 'T', 'E10+', 'K-A', 'AO', 'EC', 'RP'], dtype=object)

In [33]:
# Сортируем датафрейм по всем столбцам
df_sorted = df.sort_values(by=df.columns.tolist())

In [34]:
# Находим дубликаты
duplicates = df_sorted[df_sorted.duplicated(keep=False)]
duplicates

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
15191,Beyblade Burst,3DS,2016,role-playing,0.00,0.00,0.03,0.00,71.5,7.25,NaN
15192,Beyblade Burst,3DS,2016,role-playing,0.00,0.00,0.03,0.00,71.5,7.25,NaN
15301,11eyes: CrossOver,X360,2009,adventure,0.00,0.00,0.02,0.00,70.0,7.05,NaN
15302,11eyes: CrossOver,X360,2009,adventure,0.00,0.00,0.02,0.00,70.0,7.05,NaN
4860,18 Wheeler: American Pro Trucker,PS2,2001,racing,0.20,0.15,0.00,0.05,61.0,5.70,E
...,...,...,...,...,...,...,...,...,...,...,...
2909,Yu-Gi-Oh! The Falsebound Kingdom,GC,2002,strategy,0.49,0.13,0.07,0.02,70.0,7.80,NaN
6695,Zoo Resort 3D,3DS,2011,simulation,0.11,0.09,0.03,0.02,61.0,6.60,E
6696,Zoo Resort 3D,3DS,2011,simulation,0.11,0.09,0.03,0.02,61.0,6.60,E
8156,Zumba Fitness Rush,X360,2012,sports,0.00,0.16,0.00,0.02,73.0,6.20,E10+


Таким образом было выявлено 470 строк с дублирующимися значениями. Чтобы они не искажали результаты анализа, дубликаты необходимо удалить.

In [35]:
# Удаляем дубликаты:
df = df.drop_duplicates()

In [36]:
# Повторная проверка дубликатов:
duplicates_check = df[df.duplicated(keep=False)]
duplicates_check

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating


In [37]:
# Подсчитываем количиство строк оригинального датафрейма:
original_row_count = df_origin.shape[0]
original_row_count

16956

In [38]:
# Подсчитываем количиство строк датафрейма после всех изменений:
current_row_count = df.shape[0]
current_row_count

16446

In [39]:
# Подсчитываем количиство удалённых строк:
deleted_row_count = original_row_count - current_row_count
deleted_row_share = deleted_row_count / original_row_count
display ('Колличество удаленных строк:', deleted_row_count)
display ('Доля удалённых строк:', round(deleted_row_share, 2))

'Колличество удаленных строк:'

510

'Доля удалённых строк:'

0.03

По итогам проведённой предобработки данных была проделана следующая работа:

- значения датафрейма преобразованы к соответствующим типам данных;

- проведена работа с пропущенными значениями: удалены пропуски в столбце Year of Release, часть значений заменено на индикатор 0, пропуски в полях Critic Score и User Score заменены на медианное значение, а где это сделать было невозможно, установлено значение-индикатор -1;

- удалены дублирующиес строки.

При предобработке данных было удалено 510 строк, что составило около 3% строк датафрейма.

Таким образом, данные датафрема были подготовлены для дальнейшего анализа.

In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16446 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16444 non-null  object 
 1   platform         16446 non-null  object 
 2   year_of_release  16446 non-null  int64  
 3   genre            16444 non-null  object 
 4   na_sales         16446 non-null  float64
 5   eu_sales         16446 non-null  float64
 6   jp_sales         16446 non-null  float64
 7   other_sales      16446 non-null  float64
 8   critic_score     16446 non-null  float64
 9   user_score       16446 non-null  float64
 10  rating           9768 non-null   object 
dtypes: float64(6), int64(1), object(4)
memory usage: 1.5+ MB


In [41]:
df.head()

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,Wii Sports,Wii,2006,sports,41.36,28.96,3.77,8.45,76.0,8.0,E
1,Super Mario Bros.,NES,1985,platform,29.08,3.58,6.81,0.77,-1.0,-1.0,NaN
2,Mario Kart Wii,Wii,2008,racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009,sports,15.61,10.93,3.28,2.95,80.0,8.0,E
4,Pokemon Red/Pokemon Blue,GB,1996,role-playing,11.27,8.89,10.22,1.00,-1.0,-1.0,NaN


---

## Фильтрация данных

In [42]:
# Фильтруем данные по году выпуска игры:
df_actual = df[(df['year_of_release'] >= 2000) & (df['year_of_release'] <= 2013)].copy()

In [43]:
# Проверяем корректность применения фильтрации:
display ('Максимальное значение year_of_release:', df_actual['year_of_release'].max())
display ('Минимальное значение year_of_release:', df_actual['year_of_release'].min())

'Максимальное значение year_of_release:'

2013

'Минимальное значение year_of_release:'

2000

In [44]:
# Выводим датафрейм отсортированный по году выпуска игры:
df_actual.sort_values(by='year_of_release', ascending=False).head()

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
2229,Naruto Shippuden: Ultimate Ninja Storm 3,PS3,2013,fighting,0.32,0.32,0.15,0.15,77.0,7.90,T
10133,Dynasty Warriors 8: Xtreme Legends,PSV,2013,action,0.03,0.02,0.05,0.02,75.0,7.55,NaN
648,Tomb Raider (2013),PS3,2013,action,0.60,1.26,0.08,0.48,75.0,7.40,NaN
13720,Nobunaga's Ambition (3DS),3DS,2013,simulation,0.00,0.00,0.04,0.00,68.0,6.80,NaN
10140,Fuse (Insomniac),PS3,2013,shooter,0.06,0.04,0.00,0.02,75.0,7.40,NaN


---

## Категоризация данных

In [45]:
# Создаём категории по оценкам пользователей:
df_actual['user_score_category'] = pd.cut(df_actual['user_score'], bins=[-1, 0, 3, 8, 10], right=False, labels=["Нет оценки", "Низкая оценка", "Средняя оценка", "Высокая оценка"])

In [46]:
# Проверяем созданные категории:
df_actual['user_score_category'].unique()

['Высокая оценка', 'Средняя оценка', 'Низкая оценка', 'Нет оценки']
Categories (4, object): ['Нет оценки' < 'Низкая оценка' < 'Средняя оценка' < 'Высокая оценка']

In [47]:
# Создаём категории по оценкам критиков:
df_actual['critic_score_category'] = pd.cut(df_actual['critic_score'], bins=[-1, 0, 30, 80, 100], right=False, labels=["Нет оценки", "Низкая оценка", "Средняя оценка", "Высокая оценка"])

In [48]:
# Проверяем созданные категории:
df_actual['critic_score_category'].unique()

['Средняя оценка', 'Высокая оценка', 'Нет оценки', 'Низкая оценка']
Categories (4, object): ['Нет оценки' < 'Низкая оценка' < 'Средняя оценка' < 'Высокая оценка']

In [49]:
# Считаем количество игр по категориям оценок пользователей:
games_by_user_score_category = df_actual.groupby('user_score_category')['name'].count()
games_by_user_score_category

user_score_category
Нет оценки         114
Низкая оценка      116
Средняя оценка    9382
Высокая оценка    3169
Name: name, dtype: int64

In [50]:
# Считаем количество игр по категориям оценок критиков:
games_by_critic_score_category = df_actual.groupby('critic_score_category')['name'].count()
games_by_critic_score_category

critic_score_category
Нет оценки          242
Низкая оценка        55
Средняя оценка    10724
Высокая оценка     1760
Name: name, dtype: int64

In [51]:
# Считаем количество игр по платформам:
top_platform = df_actual.groupby('platform')['name'].count()


In [52]:
# Выводим Топ-7 платформ по количеству игр:
display('Топ-7 платформ по количеству игр:', top_platform.sort_values(ascending=False).head(7))

'Топ-7 платформ по количеству игр:'

platform
PS2     2127
DS      2120
Wii     1275
PSP     1180
X360    1121
PS3     1087
GBA      811
Name: name, dtype: int64

---

## Итоговый вывод

В рамках проекта была проделана следующая работа:

- загрузка и первичное изучение датафрейма;

- преобразование типов данных столбцов;

- поиск пропусков и их обработка;

- поиск и удаление дубликатов;

- фильтрация данных по году выпуска игр;

- категоризация игр по оценкам пользователей и критиков;

- определение лидирующих платформ по количеству игр.

По результатом проведённой работы был получен срез данных, очищенный от пропусков и дубликатов, отфильтрованный по временному периоду с 2000 г. по 2013 г. Данные приведены к соответствующим типам, созданы новые столбцы, содержащие категории оценок критиков и пользователей.

Анализ полученных данных показал следующее:

- большинство игр имеют средние оценки как и пользователей, так и критиков;

- к лидирующим платформам по количеству игр относятся: PS2, DS, Wii, PSP, X360, PS3, GBA.